# 01 — Exploratory Data Analysis

This notebook walks through the infant-cry audio dataset and produces
the interpretive plots required for the Phase 1 report.

**Sections**
1. Dataset overview & class distribution
2. Waveform visualisation (one per class)
3. Spectrogram & Mel-spectrogram inspection
4. MFCC heatmaps
5. Duration & energy statistics
6. Stationarity checks (ADF test)
7. Gaussianity checks (Shapiro–Wilk, D'Agostino)
8. Feature correlation heatmap

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display

from src.config import (
    CLASSES, CLASS_TO_IDX, IDX_TO_CLASS,
    SAMPLE_RATE, DURATION, N_MFCC, HOP_LENGTH, N_FFT,
    RAW_DIR, PLOTS_DIR,
)
from src.data_loader import load_dataset, get_class_distribution
from src.feature_extraction import extract_mfcc, extract_clip_features, extract_features_batch
from src.statistical_tests import (
    batch_stationarity_test, plot_stationarity_summary,
    batch_gaussianity_test, plot_gaussianity_summary,
    plot_feature_distributions,
)

sns.set_theme(style='whitegrid', palette='Set2')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load Dataset

In [ ]:
X_raw, y, file_paths = load_dataset()
print(f'Total clips loaded: {len(X_raw)}')
print(f'Waveform shape    : {X_raw.shape}')
print(f'Sample rate       : {SAMPLE_RATE} Hz')
print(f'Duration          : {DURATION} s')

In [ ]:
dist = get_class_distribution(y)
print('Class distribution:')
for cls, count in dist.items():
    print(f'  {cls:15s} : {count}')

In [ ]:
# Bar chart of class distribution
fig, ax = plt.subplots(figsize=(8, 5))
classes_list = list(dist.keys())
counts = list(dist.values())
bars = ax.bar(classes_list, counts, color=sns.color_palette('Set2', len(classes_list)), edgecolor='black')
ax.set_ylabel('Number of Clips')
ax.set_title('Class Distribution')
for bar, c in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(c), ha='center', fontsize=11)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'class_distribution.png', dpi=150)
plt.show()

## 2. Waveform Visualisation

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 1, figsize=(12, 3 * len(CLASSES)))
for idx, cls in enumerate(CLASSES):
    mask = np.where(y == idx)[0]
    if len(mask) == 0:
        axes[idx].set_title(f'{cls} — no samples')
        continue
    sample = X_raw[mask[0]]
    librosa.display.waveshow(sample, sr=SAMPLE_RATE, ax=axes[idx], alpha=0.7)
    axes[idx].set_title(f'Class: {cls}')
    axes[idx].set_ylabel('Amplitude')
fig.suptitle('Waveforms — One Sample per Class', fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'waveforms_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Spectrogram & Mel-Spectrogram

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 2, figsize=(14, 3 * len(CLASSES)))
for idx, cls in enumerate(CLASSES):
    mask = np.where(y == idx)[0]
    if len(mask) == 0:
        continue
    sample = X_raw[mask[0]]

    # STFT spectrogram
    S = np.abs(librosa.stft(sample, n_fft=N_FFT, hop_length=HOP_LENGTH))
    S_db = librosa.amplitude_to_db(S, ref=np.max)
    librosa.display.specshow(S_db, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                             x_axis='time', y_axis='hz', ax=axes[idx, 0])
    axes[idx, 0].set_title(f'{cls} — Spectrogram')

    # Mel spectrogram
    M = librosa.feature.melspectrogram(y=sample, sr=SAMPLE_RATE,
                                       n_fft=N_FFT, hop_length=HOP_LENGTH)
    M_db = librosa.power_to_db(M, ref=np.max)
    librosa.display.specshow(M_db, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                             x_axis='time', y_axis='mel', ax=axes[idx, 1])
    axes[idx, 1].set_title(f'{cls} — Mel-Spectrogram')

fig.suptitle('Spectrograms — One Sample per Class', fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'spectrograms_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. MFCC Heatmaps

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 1, figsize=(12, 3 * len(CLASSES)))
for idx, cls in enumerate(CLASSES):
    mask = np.where(y == idx)[0]
    if len(mask) == 0:
        continue
    sample = X_raw[mask[0]]
    mfcc = extract_mfcc(sample)
    librosa.display.specshow(mfcc, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                             x_axis='time', ax=axes[idx])
    axes[idx].set_title(f'{cls} — MFCCs ({N_MFCC} coefficients)')
    axes[idx].set_ylabel('MFCC Index')

fig.suptitle('MFCC Heatmaps — One Sample per Class', fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'mfcc_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Duration & Energy Statistics

In [ ]:
# RMS energy per clip
rms_values = []
for i in range(len(X_raw)):
    rms = np.sqrt(np.mean(X_raw[i] ** 2))
    rms_values.append(rms)
rms_values = np.array(rms_values)

df_energy = pd.DataFrame({'class': [CLASSES[label] for label in y], 'rms': rms_values})

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df_energy, x='class', y='rms', ax=ax, palette='Set2')
ax.set_title('RMS Energy Distribution per Class')
ax.set_ylabel('RMS Energy')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'rms_energy_boxplot.png', dpi=150)
plt.show()

## 6. Stationarity Checks (ADF Test)

In [ ]:
df_station = batch_stationarity_test(X_raw, y, n_samples_per_class=10)
display(df_station.head(15))

print('\nSummary — fraction of clips that are stationary (ADF p < 0.05):')
print(df_station.groupby('class')['is_stationary'].mean())

In [ ]:
plot_stationarity_summary(df_station, save=True)

**Interpretation:**  
Audio waveforms are typically stationary in the wide-sense over short windows
(20–40 ms), which is why STFT and MFCC extraction use short hop lengths.
The ADF test on the full 4-second clip often rejects the unit-root null,
confirming that the raw signal fluctuates around a stable mean — a reasonable
assumption for our frame-level feature extraction pipeline.

## 7. Gaussianity Checks

In [ ]:
# Extract clip-level features for Gaussianity tests
clip_features = extract_features_batch(X_raw, return_frame_level=False)
print(f'Clip feature matrix shape: {clip_features.shape}')

In [ ]:
df_gauss = batch_gaussianity_test(clip_features, y, n_features_to_test=10)
display(df_gauss.head(20))

In [ ]:
plot_gaussianity_summary(df_gauss, test='shapiro', save=True)
plot_gaussianity_summary(df_gauss, test='dagostino', save=True)

In [ ]:
plot_feature_distributions(clip_features, y, feature_indices=[0, 1, 2, 6, 12, 18], save=True)

**Interpretation:**  
MFCC summary statistics (especially means) tend to be approximately Gaussian
within each class — this justifies the use of GMMs, which assume the data is
generated by a mixture of Gaussians.  Higher-order statistics (skew, kurtosis)
and spectral features may deviate from normality, but the GMM's mixture
components can still model these distributions effectively.

## 8. Feature Correlation Heatmap

In [ ]:
# Correlation of first 40 features (MFCC means)
n_show = min(40, clip_features.shape[1])
corr = np.corrcoef(clip_features[:, :n_show].T)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, ax=ax)
ax.set_title(f'Feature Correlation Matrix (first {n_show} features)')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'feature_correlation_heatmap.png', dpi=150)
plt.show()

**Interpretation:**  
Adjacent MFCC coefficients show moderate positive correlation, which is
expected since they are derived from overlapping triangular filter banks.
High inter-feature correlation suggests that dimensionality reduction (PCA)
or using diagonal covariance GMMs may be beneficial.

---
## Summary

| Check | Finding |
|-------|---------|
| Class balance | *(fill after running on your data)* |
| Stationarity (ADF) | Most clips reject unit root → broadly stationary |
| Gaussianity (MFCC means) | Approximately normal within each class |
| Feature correlation | Adjacent MFCCs correlated; delta features less so |
| RMS energy | Varies across classes — useful discriminator |

These findings support the use of GMM and SVM on MFCC-based features.